# Exploring the HuggingFace Transformers Low-Level API

This notebook dives into the lower-level API of the HuggingFace `transformers` library, moving beyond the high-level `pipeline()` abstraction to work directly with **tokenizers** and **model objects**.

## What this notebook covers

- Loading and inspecting quantized LLMs (Llama, Phi, Gemma, Qwen, DeepSeek)
- Using `AutoTokenizer` and `AutoModelForCausalLM` directly
- Applying chat templates and understanding tokenized inputs
- 4-bit quantization with `BitsAndBytesConfig` to reduce GPU memory usage
- Streaming model output with `TextStreamer`
- Inspecting transformer architecture layers (embeddings, decoder blocks, LM head)
- Proper GPU memory cleanup between model loads

## Environment

Designed to run on a **free or low-cost Google Colab T4 GPU** runtime.

> **Note:** If you encounter a `CUDA is required but not available for bitsandbytes` error mid-run, it usually means Colab swapped your runtime. Fix it by going to **Runtime → Disconnect and delete runtime**, reconnecting to a fresh T4, and re-running cells from the top.

## Setup

Install required packages. `bitsandbytes` enables quantization, `accelerate` handles device placement, and `transformers` provides the model and tokenizer APIs.

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

## Authentication

A HuggingFace account and API token are required to download gated models like Llama.

1. Create a free account at [huggingface.co](https://huggingface.co) if you haven't already
2. Go to **Settings → Access Tokens** and create a token with **Write** permissions
3. In Colab, click the 🔑 **Secrets** icon in the left panel and add: `HF_TOKEN = <your_token>`
4. Run the cell below to authenticate

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

## Model Selection

This notebook experiments with several open-source instruction-tuned models:

| Model | Provider | Notes |
|---|---|---|
| `Llama-3.2-1B-Instruct` | Meta | Small, fast; requires accepting Meta's ToS at [HF model page](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) |
| `Phi-4-mini-instruct` | Microsoft | Efficient small model |
| `gemma-3-270m-it` | Google | Tiny but capable; requires accepting Google's ToS at [HF model page](https://huggingface.co/google/gemma-3-270m-it) |
| `Qwen3-4B-Instruct` | Alibaba | Strong multilingual model |
| `DeepSeek-R1-Distill-Qwen-1.5B` | DeepSeek | Reasoning-focused distilled model |

> To use `Meta-Llama-3.1-8B-Instruct` instead (larger, more capable), uncomment that line below. Approval for any 3.1 model covers the whole Llama 3.1 family.

In [ ]:
# Uncomment to use the larger Llama 3.1 8B model instead:
# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

LLAMA    = "meta-llama/Llama-3.2-1B-Instruct"
PHI      = "microsoft/Phi-4-mini-instruct"
GEMMA    = "google/gemma-3-270m-it"
QWEN     = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [ ]:
# Define a test prompt — used across all models for easy comparison
messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
  ]

## 4-Bit Quantization

Loading large models on a free T4 GPU (16 GB VRAM) requires reducing memory usage. We use **4-bit NF4 quantization** via `BitsAndBytesConfig`:

- `load_in_4bit`: stores weights in 4-bit instead of 16/32-bit
- `bnb_4bit_use_double_quant`: applies a second quantization pass on the quantization constants for additional savings
- `bnb_4bit_compute_dtype=bfloat16`: uses bfloat16 for the actual matrix math (better numerical stability than float16)
- `bnb_4bit_quant_type="nf4"`: uses the Normal Float 4 data type, optimized for normally-distributed weights

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

## Part 1: Tokenizer and Raw Model Output

Before using convenience wrappers, let's manually walk through the inference pipeline to understand what's happening at each step:

1. **Tokenizer** converts the chat messages into token IDs using the model's chat template
2. **Model** generates new token IDs from the input
3. **Decoder** converts the output token IDs back into readable text

> If you hit a `403` permissions error on the Llama model, visit [the model page](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B) and accept Meta's terms of service.

In [ ]:
# Step 1: Tokenize — convert messages into input token IDs using the model's chat template
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")

In [ ]:
# Inspect the raw token IDs — this is what the model actually sees
inputs

In [ ]:
# Step 2: Load the model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

In [ ]:
# How much GPU memory is the quantized model using?
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprint: {memory:,.1f} MB")

## Inspecting the Transformer Architecture

Printing the model object reveals the full neural network architecture. Key things to look for:

- **Embedding layer**: maps token IDs to high-dimensional vectors (e.g., 4,096-dim for Llama 3.1 8B)
- **Decoder layers**: repeated blocks (16 for the 1B model, 32 for the 8B), each containing:
  - Self-attention layers (how tokens attend to each other)
  - MLP layers (feed-forward transformation)
  - RMS normalization layers
- **LM Head**: a linear layer that projects the final hidden state to vocabulary logits, producing the next-token probability distribution

The quantized weights will appear as `Linear4bit` layers rather than standard `Linear` layers.

> Want to go deeper? The full PyTorch implementation of Llama is readable on GitHub:  
> https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

In [ ]:
# Print the full model architecture — explore the layers
model

In [ ]:
# Step 3: Generate new tokens from the input
outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

In [ ]:
# Step 4: Decode the output token IDs back into human-readable text
tokenizer.decode(outputs[0])

In [ ]:
# Free GPU memory before loading the next model
# The memory may not drop immediately in the Resources panel, but it is released for subsequent use
del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

## Part 2: Reusable `generate()` Function with Streaming

Now we wrap everything into a clean function. Two improvements over the manual approach above:

1. **`TextStreamer`**: tokens are printed to the console as they're generated, rather than waiting for the full output — this gives a real-time feel similar to ChatGPT
2. **`add_generation_prompt=True`**: tells the chat template to append the assistant turn opener, prompting the model to generate a *response* rather than continuing the user message

The `quant` flag lets us skip quantization for small models (like Gemma 270M) that fit in memory without it.

In [ ]:
def generate(model, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
    streamer = TextStreamer(tokenizer)
    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
    outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)

## Running Multiple Models

Using the same prompt across different models makes it easy to compare their behavior, tone, and reasoning style.

In [ ]:
# Microsoft Phi-4 Mini
generate(PHI, messages)

In [ ]:
# Google Gemma 3 270M — small enough to run without quantization
# Requires accepting Google's terms: https://huggingface.co/google/gemma-3-270m-it
messages_gemma = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages_gemma, quant=False)

In [ ]:
# Alibaba Qwen3 4B
generate(QWEN, messages)

In [ ]:
# DeepSeek R1 Distill — a reasoning model; uses more tokens to "think" before answering
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)

## Key Takeaways

- The HuggingFace `transformers` low-level API (`AutoTokenizer` + `AutoModelForCausalLM`) gives full control over the inference pipeline
- **Chat templates** are essential for instruction-tuned models — they format the conversation in the exact way each model was trained to expect
- **4-bit quantization** makes it practical to run 1B–8B parameter models on a single consumer GPU
- **Streaming** via `TextStreamer` significantly improves the perceived interactivity of model outputs
- Inspecting the model object is a useful way to build intuition for transformer architecture without diving into the PyTorch source code